In [1]:
import numpy as np
import pandas as pd
from copy import copy

In [2]:
df=pd.read_csv('data/iris.csv')

In [3]:
X,y=df.loc[:99,['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']],df.loc[:99,'Species']

In [4]:
y=y.map({'Iris-setosa':-1,'Iris-versicolor':1})

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

In [6]:
class AdaBoost:
    def __init__(self,classifier,n_classifiers):
        self.classifier=classifier
        self.n_classifiers=n_classifiers
        self.classifiers=[]
        self.alphas=[]
    def fit(self,X,y):
        self.X=np.asarray(X)
        self.y=np.asarray(y)
        m,n=self.X.shape
        self.weights=np.ones(m)/m       
        for _ in range(self.n_classifiers):
            clf=copy(self.classifier)
            idxs = np.random.choice(m, size=m, replace=True, p=self.weights)
            X_sampled = self.X[idxs]
            y_sampled = self.y[idxs]
            clf.fit(X_sampled,y_sampled)
            preds=clf.predict(self.X)
            error=np.sum(self.weights[preds!=self.y])
            alpha=0.5*np.log((1-error+ 1e-10)/(error+ 1e-10))
            self.alphas.append(alpha)
            self.weights=self.weights*np.exp(-alpha*preds*self.y)
            self.weights=self.weights/np.sum(self.weights)
            self.classifiers.append(clf)
    def predict(self,X):
        X=np.asarray(X)
        all_preds=np.array([alpha*clf.predict(X) for alpha,clf in zip(self.alphas,self.classifiers)])
        return np.sign(np.sum(all_preds,axis=0))

In [7]:
from models import DecisionTree

In [8]:
dt=DecisionTree(max_depth=2,min_samples=5)
ab=AdaBoost(dt,10)
ab.fit(X_train,y_train)
preds=ab.predict(X_test)
print("Accuracy:",np.mean(preds==y_test)*100,'%')

Accuracy: 100.0 %
